<a href="https://colab.research.google.com/github/sergi-villanueva/Xatbot-1.4---Sergi-Villanueva/blob/main/XatBot_talent_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
cat > XatBot_talent_2026.ipynb << 'EOF'
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 🧠 XatBot Talent 2026 - Sergi Villanueva\n",
    "**Chatbot dinàmic per WordPress amb Gemini + Scraping**\n",
    "\n",
    "Aquest notebook està dissenyat per executar-se a **Google Colab**."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Instal·lació de llibreries\n",
    "!pip install google-generativeai requests beautifulsoup4 flask pyngrok -q"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import google.generativeai as genai\n",
    "import requests\n",
    "from bs4 import BeautifulSoup\n",
    "from flask import Flask, request, jsonify\n",
    "from pyngrok import ngrok\n",
    "import os\n",
    "import time\n",
    "\n",
    "# ================= CONFIGURACIÓ =================\n",
    "GEMINI_API_KEY = \"PEGA_LA_TEVA_CLAU_AQUI\"   # Obté-la a: https://aistudio.google.com/app/apikey\n",
    "WORDPRESS_URL = \"https://www.elteuweb.com\"   # ← Canvia per la URL del teu WordPress\n",
    "\n",
    "genai.configure(api_key=GEMINI_API_KEY)\n",
    "model = genai.GenerativeModel('gemini-2.0-flash-exp')  # o gemini-1.5-flash\n",
    "\n",
    "app = Flask(__name__)\n",
    "\n",
    "def scrape_wordpress():\n",
    "    \"\"\"Extreu informació actual de la web cada vegada\"\"\"\n",
    "    try:\n",
    "        headers = {'User-Agent': 'Mozilla/5.0'}\n",
    "        r = requests.get(WORDPRESS_URL, headers=headers, timeout=15)\n",
    "        soup = BeautifulSoup(r.text, 'html.parser')\n",
    "        \n",
    "        # Agafa contingut rellevant\n",
    "        texts = []\n",
    "        for tag in soup.find_all(['h1', 'h2', 'h3', 'p', 'li', 'strong']):\n",
    "            text = tag.get_text(strip=True)\n",
    "            if len(text) > 10:\n",
    "                texts.append(text)\n",
    "        \n",
    "        context = \" \".join(texts[:40])\n",
    "        return context[:8000]  # Límit de tokens\n",
    "    except Exception as e:\n",
    "        return f\"Error carregant la web: {str(e)}\"\n",
    "\n",
    "@app.route('/chat', methods=['POST'])\n",
    "def chat():\n",
    "    try:\n",
    "        user_message = request.json.get('message', '')\n",
    "        context = scrape_wordpress()\n",
    "        \n",
    "        prompt = f\"\"\"Ets un assistent expert i professional del negoci de Sergi Villanueva.\n",
    "        Informació actual de la pàgina web:\n",
    "        {context}\n",
    "\n",
    "        Respon sempre en català, de forma clara, amable i professional.\n",
    "        Pregunta de l'usuari: {user_message}\"\"\"\n",
    "        \n",
    "        response = model.generate_content(prompt)\n",
    "        return jsonify({\"reply\": response.text})\n",
    "    except Exception as e:\n",
    "        return jsonify({\"reply\": \"Ho sento, hi ha hagut un error. Torna-ho a provar.\"})\n",
    "\n",
    "if __name__ == \"__main__\":\n",
    "    # Inicia ngrok\n",
    "    public_url = ngrok.connect(5000)\n",
    "    print(\"✅ XatBot en funcionament!\")\n",
    "    print(f\"🔗 URL pública: {public_url}\")\n",
    "    print(\"\\nCopia aquesta URL i posa-la al Widget_talent.html\")\n",
    "    \n",
    "    app.run(port=5000)"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},
  "language_info": {"name": "python"}
 },
 "nbformat": 4,
 "nbformat_minor": 4
}
EOF